# Wind Load Calc — Sign on a Vandal Protection Fence on a Bridge Railing

A 3 ft × 5 ft solid metal sign clamped to a post of an **ODOT Vandal Protection
Fence** (SCD VPF-1-24 / legacy VPF-1-90) mounted on a concrete bridge railing.
We chase the wind force through the whole load path and report demand/capacity
at every link:

1. **Wind load** — AASHTO *LRFD Specifications for Structural Supports for
   Highway Signs, Luminaires, and Traffic Signals* (LRFDLTS-1), Article 3.8
2. **Sign-to-fence connection** — U-bolt clamps (LTS 5.15 → AISC bolt shear)
3. **Fence post** — cantilever flexure (LTS Section 5)
4. **Fabric-to-post connection** — tension bands per VPF general note (7)
5. **Anchor bolts** — adhesive anchors per VPF note (5) →
   `civilpy.structural.concrete.AnchorBolts` (ACI 318-19 Ch. 17, which is where
   LTS 5.16.3 and the VPF drawings both send anchorage design)

Fence hardware is taken from the drawings, not assumed: **6'-0" straight fence**
(VPF-1-24 — the 8'-0" straight option existed on VPF-1-90 but the current
standard is 6'), 1-in 11-ga mesh, one tension band per foot of fabric, and the
standard base plates.  Load combination: **Extreme I** — 1.0·W with the 700-yr
MRI wind map, 1.1/0.9·DC (LTS Table 3.4-1, Table 3.8-1).

## Geometry and hardware (from the SCDs)

| item | value | source |
|---|---|---|
| fence fabric height, $H_f$ | 6'-0" | VPF-1-24 post section PS-2 |
| fabric | 1-in diamond mesh, 0.120-in (11-ga) wire | VPF note (16) |
| line/end posts | 2.880-in OD Grade 2 pipe, Fy = 50 ksi, 4.64 lb/ft | VPF note (1) |
| rails (top/line/bottom) | 1.660-in OD Grade 2 pipe, Fy = 50 ksi | VPF note (2) |
| post spacing | 10'-0" max with BP-1 (5'-0" with BP-2) | VPF-1-24 sheet 2 |
| tension bands | ⅛×1-in, one per ft of fabric, ⅜-in bolts | VPF note (7) |
| base plate (primary case) | VPF-1-90 BP-1: 8½×13×1-in flat plate | VPF-1-90 sheet 3 |
| anchors (primary case) | 4 × ½-in ASTM A193 B7 adhesive rods, 7-in min embed | VPF-1-90 notes (4)/(5) |
| adhesive | Hilti HIT-HY 200 (first listed approved product) | ICC-ES ESR-3187 |
| railing | BR-type parapet, 12-in top width, f′c ≥ 4,000 psi | VPF-1-90 sheet 2, note (22) |
| sign | 3'T × 5'W solid panel, top at top of fence, centered on one post | design case |
| site | Ohio — V(700-yr) = 115 mph, V(10-yr, Fig. 3.8-4) = 76 mph | LTS Figs. 3.8-1b/3.8-4 |

Risk category is *Typical* (a failed post/sign could drop into the travelway),
so Table 3.8-1 gives MRI = 700 yrs; the "roadside sign supports: 10-yr" relief
does not apply.  VPF note (22) requires a **special design** when f′c < 4,000
psi or the fence midpoint sits more than 50 ft above the surrounding terrain —
this calc *is* that special-design check for the sign retrofit; we keep the
fence below the 50-ft trigger so Kz rides its 16-ft floor.

The current VPF-1-24 base plates are all bent (wrap-around) plates with ⅝-in
F1554 Gr 36 rods (verticals 6-in embed, horizontal 4-in); the flat 8½×13 plate
above is the legacy VPF-1-90 detail found on existing bridges — the likely
candidates for a sign retrofit — and is run as the primary case.  The
wrap-plate interface is checked second, against ODOT's own published design
load.

In [1]:
import math
import numpy as np
from civilpy.general import units

# --- site / spec inputs -------------------------------------------------
V   = 115.0        # mph, 3-s gust, 700-yr MRI (LTS Fig. 3.8-1b, Ohio)
V10 = 76.0         # mph, 10-yr MRI service wind (LTS Fig. 3.8-4)

# --- fence geometry (VPF-1-24) -------------------------------------------
H_fabric = 6.0            # ft, fabric height above the base plate
s_post   = 10.0           # ft, max post spacing with BP-1
d_post   = 2.880 / 12.0   # ft, post OD
t_post   = 0.160          # in, Grade 2 pipe wall (4.64 plf checks below)
d_rail   = 1.660 / 12.0   # ft, rail OD
n_rails  = 3              # top, line (mid-height), bottom

# fabric solidity: 1-in diamond mesh of 0.120-in wire has two wire runs
# crossing each 1-in cell -> projected solid fraction ~ 2*d/mesh
wire_dia, mesh = 0.120, 1.0
solidity = 2 * wire_dia / mesh
print(f"fabric solidity (1-in mesh, 11-ga) ~ {solidity:.2f}")

# --- sign ----------------------------------------------------------------
sign_w, sign_h = 5.0, 3.0            # ft
A_sign = sign_w * sign_h
z_sign_cg = H_fabric - sign_h / 2.0  # centroid above base plate, ft
print(f"sign {A_sign:.0f} ft^2, centroid {z_sign_cg:.1f} ft above the base plate")

fabric solidity (1-in mesh, 11-ga) ~ 0.24
sign 15 ft^2, centroid 4.5 ft above the base plate


## Step 1 — Design wind pressure (LTS Article 3.8)

$$P_z = 0.00256\,K_z\,K_d\,G\,V^2\,C_d \quad \text{(psf, Eq. 3.8.1-1)}$$

- $K_z$ — Eq. 3.8.4-1 (Exposure C: $\alpha=9.5$, $z_g=900$ ft, $z\ge16$ ft)
- $K_d$ — Table 3.8.5-1: round pole → 0.95
- $G$ — Art. 3.8.6: 1.14 minimum
- $C_d$ — Table 3.8.7-1: sign panel interpolated on $L_{sign}/W_{sign}$;
  cylindrical members by the $C_vVd$ regime ($C_v=0.8$ at the extreme limit
  state).  Chain-link fabric is not tabulated — note *a* permits established
  values for untabulated shapes, so the fabric is treated as its net (solid)
  area with $C_d = 1.2$ (CLFMI wind-load-guide practice).

In [2]:
def Kz(z_ft):
    """LTS Eq. 3.8.4-1, Exposure C, with the 16-ft floor."""
    z = max(z_ft, 16.0)
    return 2.0 * (z / 900.0) ** (2.0 / 9.5)

Kd = 0.95      # Table 3.8.5-1, round pole
G  = 1.14      # Art. 3.8.6

def Cd_cylinder(d_ft, Cv=0.8):
    """Table 3.8.7-1, single cylindrical member."""
    cvvd = Cv * V * d_ft
    if cvvd <= 39.0:
        return 1.10
    if cvvd < 78.0:
        return 129.0 / cvvd**1.3
    return 0.45

# sign panel Cd by aspect ratio (Table 3.8.7-1)
ratios, cds = [1.0, 2.0, 5.0, 10.0, 15.0], [1.12, 1.19, 1.20, 1.23, 1.30]
Cd_sign   = float(np.interp(sign_w / sign_h, ratios, cds))
Cd_fabric = 1.2
Cd_post   = Cd_cylinder(d_post)
Cd_rail   = Cd_cylinder(d_rail)

kz = Kz(16.0)                          # fence top well under the 16-ft floor height
q  = 0.00256 * kz * Kd * G * V**2      # psf per unit Cd
print(f"Kz = {kz:.3f} ->  q = {q:.1f} psf per unit Cd")
print(f"Cd: sign {Cd_sign:.3f} (L/W = {sign_w/sign_h:.2f}) | fabric 1.2 | post {Cd_post:.2f} | rail {Cd_rail:.2f}")
print(f"Pz: sign {q*Cd_sign:.1f} psf | fabric (net) {q*Cd_fabric:.1f} psf | members {q*Cd_post:.1f} psf")

Kz = 0.856 ->  q = 31.4 psf per unit Cd
Cd: sign 1.167 (L/W = 1.67) | fabric 1.2 | post 1.10 | rail 1.10
Pz: sign 36.6 psf | fabric (net) 37.7 psf | members 34.5 psf


In [3]:
# Factored member-end forces, Extreme I (gamma_W = 1.0) --------------------
# The sign shadows its own patch of fabric; rails average out at mid-height.
F_sign   = q * Cd_sign   * A_sign
A_fab_net_sign = (s_post * H_fabric - A_sign) * solidity      # sign post bay
A_fab_net_line = (s_post * H_fabric) * solidity               # plain bay
F_fab_s  = q * Cd_fabric * A_fab_net_sign
F_fab_l  = q * Cd_fabric * A_fab_net_line
F_post   = q * Cd_post   * (d_post * H_fabric)
F_rails  = q * Cd_rail   * (n_rails * s_post * d_rail)

# sign post
V_sign_post = F_sign + F_fab_s + F_post + F_rails
M_sign_post = (F_sign * z_sign_cg
               + (F_fab_s + F_post + F_rails) * H_fabric / 2) * 12   # lb-in

# plain line post
V_line = F_fab_l + F_post + F_rails
M_line = V_line * H_fabric / 2 * 12

print(f"F_sign = {F_sign:5.0f} lb @ {z_sign_cg:.1f} ft   F_fabric = {F_fab_s:.0f} lb   "
      f"F_post = {F_post:.0f} lb   F_rails = {F_rails:.0f} lb")
print(f"sign post:  V = {V_sign_post:5.0f} lb   M = {M_sign_post/1000:5.1f} kip-in")
print(f"line post:  V = {V_line:5.0f} lb   M = {M_line/1000:5.1f} kip-in")

F_sign =   549 lb @ 4.5 ft   F_fabric = 407 lb   F_post = 50 lb   F_rails = 143 lb
sign post:  V =  1149 lb   M =  51.3 kip-in
line post:  V =   736 lb   M =  26.5 kip-in


## Step 2 — Sign-to-fence connection

The panel is clamped to the sign post with **two ⅜-in U-bolts** (4 shear legs),
matching the fence's own ⅜-in hardware (VPF notes 7/8).  LTS 5.15 sends bolted
connections to AISC: $\phi R_n = 0.75\,F_{nv}A_b$, $F_{nv}=27$ ksi (A307-class,
threads included).

In [4]:
n_legs = 4
V_leg = F_sign / n_legs
A_b38 = math.pi * 0.375**2 / 4
phiRn_leg = 0.75 * 27_000 * A_b38
dc_ubolt = V_leg / phiRn_leg
print(f"shear/leg = {V_leg:.0f} lb  vs  phiRn = {phiRn_leg:.0f} lb  ->  D/C = {dc_ubolt:.2f}")

shear/leg = 137 lb  vs  phiRn = 2237 lb  ->  D/C = 0.06


## Step 3 — Fence post flexure

Posts are the drawing's actual pipe — 2.880-in OD × 0.160-in wall Grade 2
(Fy = 50 ksi, 4.64 plf) — *not* Sch 40, so we compute section properties
directly.  $\phi M_n = 0.9\,F_y\,S$ (compact round tube; using S over Z is
slightly conservative).

In [5]:
Fy_post = 50_000.0     # psi, Grade 2 pipe per VPF note (1)

OD, t = 2.880, t_post
ID = OD - 2*t
A_p = math.pi/4 * (OD**2 - ID**2)
I_p = math.pi/64 * (OD**4 - ID**4)
S_p = I_p / (OD/2)
print(f"post section: A = {A_p:.3f} in^2, I = {I_p:.3f} in^4, S = {S_p:.3f} in^3, "
      f"weight = {A_p*3.4:.2f} plf (drawing says 4.64)")

phiMn_post = 0.9 * Fy_post * S_p
dc_line  = M_line / phiMn_post
dc_signp = M_sign_post / phiMn_post
print(f"\nline post:  M = {M_line/1000:5.1f} kip-in vs phiMn = {phiMn_post/1000:.1f} kip-in -> D/C = {dc_line:.2f}")
print(f"sign post:  M = {M_sign_post/1000:5.1f} kip-in vs phiMn = {phiMn_post/1000:.1f} kip-in -> D/C = {dc_signp:.2f}"
      + ("   <-- FAILS" if dc_signp > 1 else ""))

post section: A = 1.367 in^2, I = 1.269 in^4, S = 0.881 in^3, weight = 4.65 plf (drawing says 4.64)

line post:  M =  26.5 kip-in vs phiMn = 39.6 kip-in -> D/C = 0.67
sign post:  M =  51.3 kip-in vs phiMn = 39.6 kip-in -> D/C = 1.29   <-- FAILS


**First weak spot:** the standard VPF post carries the bare fence with margin,
but hanging the 15-ft² sign on it overloads it.  Practical fixes: a heavier
post at the sign location (below), dropping to BP-2's 5-ft spacing *and* a
smaller sign, or keeping signs off the fence entirely.

A common upsize — NPS 4 Sch 40 (A53 Gr B, Fy = 35 ksi) in place of the fence
post; the 3½-in VPF post sleeve geometry already anticipates a larger tube at
base plates:

In [6]:
from civilpy.structural.steel import SteelSection

big = SteelSection("Pipe4SCH40")
S_big = big.S_x.magnitude
phiMn_big = 0.9 * 35_000 * S_big
dc_big = M_sign_post / phiMn_big
print(f"Pipe4SCH40: phiMn = {phiMn_big/1000:.1f} kip-in -> D/C = {dc_big:.2f}")

# Service I deflection at the 10-yr wind (LTS Fig. 3.8-4)
E = 29_000_000.0
scale10 = (V10 / V) ** 2
z_res = M_sign_post / V_sign_post / 12          # resultant height, ft
P_serv = V_sign_post * scale10
delta = P_serv * (z_res*12)**3 / (3 * E * big.I_x.magnitude)
print(f"10-yr wind: {P_serv:.0f} lb at {z_res:.1f} ft -> tip deflection ~ {delta:.2f} in")

Pipe4SCH40: phiMn = 95.4 kip-in -> D/C = 0.54
10-yr wind: 502 lb at 3.7 ft -> tip deflection ~ 0.08 in


## Step 4 — Fabric-to-post connection

Per VPF note (7): **one ⅛×1-in tension band per foot of fabric height** — six
bands on the 6-ft fence — each with a ⅜-in galvanized bolt in single shear.
(Fabric ties, note 11, carry the same rhythm on line posts.)

In [7]:
n_bands = int(H_fabric)          # one per foot of fabric height
V_band = F_fab_s / n_bands
phiRn_band = 0.75 * 27_000 * A_b38
dc_band = V_band / phiRn_band
print(f"{n_bands} bands -> {V_band:.0f} lb/bolt vs phiRn = {phiRn_band:.0f} lb -> D/C = {dc_band:.2f}")
print(f"(average net fabric pressure is only {F_fab_s/(s_post*H_fabric):.1f} psf gross — "
      "the 11-ga fabric itself is nowhere near its breaking strength)")

6 bands -> 68 lb/bolt vs phiRn = 2237 lb -> D/C = 0.03
(average net fabric pressure is only 6.8 psf gross — the 11-ga fabric itself is nowhere near its breaking strength)


## Step 5 — Anchor rods (VPF-1-90 BP-1 flat plate, adhesive anchors)

The legacy flat plate: **8½ × 13 × 1-in**, four **½-in ASTM A193 B7 threaded
rods** set with adhesive at **7-in min embedment** into the 12-in-wide BR
parapet top.  Rod columns sit 3 in and 7 in from the back face (4-in gage
across the parapet, 10-in rows along it); the post is offset ~2½ in from the
back edge.

Adhesive design values from **ICC-ES ESR-3187** (Hilti HIT-HY 200, the first
approved product on both VPF drawings), hammer-drilled, dry, temp range A:

| parameter | value | ESR table |
|---|---|---|
| τ_k,cr / τ_k,uncr (½-in rod) | 1,135 / 2,220 psi | Table 14 |
| k_c (cracked / uncracked) | 17 / 24 | Table 12 |
| φ tension / shear, concrete modes (Cond. B) | 0.65 / 0.70 | Table 12 |
| ½-in B7 rod: N_sa / V_sa | 17,735 / 10,640 lb (φ = 0.75/0.65) | Table 10 |
| permitted h_ef, ½-in | 2¾ – 10 in | Table 12 |

The parapet top is treated as **cracked** concrete (conservative, and the VPF
notes require products qualified for cracked concrete).  The base moment
resolves into a couple across the 4-in bolt gage; wind can blow either way, so
the tension pair is taken at the worst column — the back column with just 3 in
of edge distance.

In [8]:
gage = 4.0                                  # in, across the parapet
T_pair = M_sign_post / gage                 # lb on the 2-rod tension column
V_group = V_sign_post
print(f"tension couple T = M/gage = {M_sign_post/1000:.1f} kip-in / {gage:.0f} in = "
      f"{T_pair:.0f} lb on the tension column ({T_pair/2:.0f} lb/rod)")
print(f"group shear V = {V_group:.0f} lb ({V_group/4:.0f} lb/rod)")

tension couple T = M/gage = 51.3 kip-in / 4 in = 12816 lb on the tension column (6408 lb/rod)
group shear V = 1149 lb (287 lb/rod)


In [9]:
from civilpy.structural.concrete import AnchorBolts

# projected breakout area of the tension column, actual parapet geometry
# (ACI 17.6.2.1.1): edges only across the 12-in width; barrier runs long.
h_ef, c_back, c_road, s_row = 7.0, 3.0, 9.0, 10.0
x_ext = min(c_back, 1.5*h_ef) + min(c_road, 1.5*h_ef)         # across parapet
y_ext = 1.5*h_ef + s_row + 1.5*h_ef                            # along parapet
A_Nc_hand = x_ext * y_ext
print(f"hand A_Nc = {x_ext:.1f} x {y_ext:.1f} = {A_Nc_hand:.0f} in^2 "
      f"(vs A_Nco = {(3*h_ef)**2:.0f} per anchor)")

anchors = AnchorBolts(
    f_c=4000.0, h_a=30.0,                      # parapet depth below the top
    d_a=0.5, h_ef=h_ef,
    f_ya=105_000.0, f_uta=125_000.0,           # ASTM A193 B7 (ESR-3187 Table 10)
    n_x=1, n_y=2, s_x=0.0, s_y=s_row,          # the tension column
    c_a1=c_back, c_a2=100.0,
    A_Nc=A_Nc_hand,
    anchor_type="adhesive",
    tau_cr=1135.0, tau_uncr=2220.0,            # ESR-3187 Table 14, 1/2-in rod
    is_cracked=True, has_supp_reinf=False,
    N_ua=T_pair, V_ua=V_group / 2.0,           # column's share of shear
    shear_direction="perpendicular",
)
print(anchors.summary())

hand A_Nc = 12.0 x 31.0 = 372 in^2 (vs A_Nco = 441 per anchor)
-----------------------------------------------------------------------------------------------------
Limit State                         Ref                           φSn (kip)     Demand     DCR Status
-----------------------------------------------------------------------------------------------------
Steel strength (tension)            ACI 318-19 Eq. 17.6.1.2           26.61      12.82   0.482     OK
Concrete breakout (tension)         ACI 318-19 Eq. 17.6.2.1b           9.24      12.82   1.387     NG
Bond strength (tension)             ACI 318-19 Eq. 17.6.5.1.1b         7.13      12.82   1.798     NG
Steel strength (shear)              ACI 318-19 Eq. 17.7.1.2b          13.84       0.57   0.042     OK
Concrete breakout (shear)           ACI 318-19 Eq. 17.7.2.1b           3.64       0.57   0.158     OK
Pryout (shear)                      ACI 318-19 Eq. 17.7.3.1           18.48       0.57   0.031     OK
Tension-shear inter

*(The `AnchorBolts` default projected areas clip to `c_a_min` on **all**
sides — correct for a pedestal, conservative on a long parapet — so the
tension-breakout area above is hand-computed per ACI 17.6.2.1.1 and passed as
an override.  Bond influence areas inside the class carry the same
conservatism; the printed bond DCR is therefore a lower bound on capacity.)*

Run the same anchorage under the **bare fence** (no sign) for comparison:

In [10]:
T_pair_fence = M_line / gage
anchors_fence = AnchorBolts(
    f_c=4000.0, h_a=30.0, d_a=0.5, h_ef=h_ef,
    f_ya=105_000.0, f_uta=125_000.0,
    n_x=1, n_y=2, s_x=0.0, s_y=s_row, c_a1=c_back, c_a2=100.0,
    A_Nc=A_Nc_hand, anchor_type="adhesive",
    tau_cr=1135.0, tau_uncr=2220.0, is_cracked=True,
    N_ua=T_pair_fence, V_ua=V_line / 2.0,
    shear_direction="perpendicular",
)
worst_sign  = max(r.dcr for r in anchors.check_all().values())
worst_fence = max(r.dcr for r in anchors_fence.check_all().values())
print(f"bare fence  (T = {T_pair_fence:5.0f} lb): worst anchor DCR = {worst_fence:.2f}")
print(f"fence + sign (T = {T_pair:5.0f} lb): worst anchor DCR = {worst_sign:.2f}")

bare fence  (T =  6620 lb): worst anchor DCR = 0.93
fence + sign (T = 12816 lb): worst anchor DCR = 1.80


**Second (governing) weak spot: the concrete side of the anchorage.**  The
standard detail is comfortable under the fence it was designed for, but the
sign roughly doubles the base moment and the concrete limit states (breakout /
bond on 3-in edge distance) blow through 1.0 — while the B7 rods themselves
stay far below their steel capacity.  Adding or upsizing rods does nothing;
concrete governs.  That is also exactly the LTS 5.16.3 ductility complaint:
the steel cannot reach its strength before the concrete lets go.

## Cross-check — the wrap-around plate and ODOT's published design load

VPF-1-90's general notes state the factored design load for the two-anchor
**horizontal** connection of the wrap-around BP-3 plate: **7.1 kips tension +
1.4 kips shear**.  That is ODOT telling us the demand envelope the standard
fence was designed to.  Resolving our post base moment into the wrap-plate's
horizontal pair (bearing at the top corner, anchors ~5 in below it):

In [11]:
lever = 5.0     # in, top-corner bearing to horizontal anchor line (BP-3 end view, MIN.)
T_wrap_fence = M_line / lever
T_wrap_sign  = M_sign_post / lever
print(f"bare fence:   T = {T_wrap_fence/1000:.1f} kips vs ODOT design envelope 7.1 kips "
      f"({T_wrap_fence/7100:.0%})")
print(f"fence + sign: T = {T_wrap_sign/1000:.1f} kips vs 7.1 kips ({T_wrap_sign/7100:.0%})"
      + ("   <-- exceeds the standard detail's design basis" if T_wrap_sign > 7100 else ""))

bare fence:   T = 5.3 kips vs ODOT design envelope 7.1 kips (75%)
fence + sign: T = 10.3 kips vs 7.1 kips (144%)   <-- exceeds the standard detail's design basis


The bare fence sits inside ODOT's envelope — consistent with the drawing
being a standard — and the sign pushes past it.  Same conclusion from the
other direction, using ODOT's own number instead of ours.

## Load-path summary

In [12]:
import pandas as pd

rows = [
    ("1. wind pressure (LTS 3.8)",       f"{q*Cd_sign:.1f} psf on sign", "-", None),
    ("2. sign U-bolt clamps",            f"{V_leg:.0f} lb/leg",  f"{phiRn_leg:.0f} lb", dc_ubolt),
    ("3a. line post (bare fence)",       f"{M_line/1000:.1f} k-in", f"{phiMn_post/1000:.1f} k-in", dc_line),
    ("3b. sign post (std VPF post)",     f"{M_sign_post/1000:.1f} k-in", f"{phiMn_post/1000:.1f} k-in", dc_signp),
    ("3c. sign post (Pipe4 upsize)",     f"{M_sign_post/1000:.1f} k-in", f"{phiMn_big/1000:.1f} k-in", dc_big),
    ("4. fabric tension-band bolts",     f"{V_band:.0f} lb", f"{phiRn_band:.0f} lb", dc_band),
    ("5a. anchors, bare fence",          f"{T_pair_fence:.0f} lb", "concrete governs", worst_fence),
    ("5b. anchors, fence + sign",        f"{T_pair:.0f} lb", "concrete governs", worst_sign),
    ("5c. wrap plate vs ODOT 7.1k",      f"{T_wrap_sign/1000:.1f} kips", "7.1 kips", T_wrap_sign/7100),
]
df = pd.DataFrame(rows, columns=["load-path link", "demand", "capacity", "D/C"])
df["status"] = df["D/C"].apply(lambda x: "-" if pd.isna(x) else ("OK" if x <= 1.0 else "NG"))
df["D/C"] = df["D/C"].apply(lambda x: "-" if pd.isna(x) else f"{x:.2f}")
df

,load-path link,demand,capacity,D/C,status
0,1. wind pressure (LTS 3.8),36.6 psf on sign,-,-,-
1,2. sign U-bolt clamps,137 lb/leg,2237 lb,0.06,OK
2,3a. line post (bare fence),26.5 k-in,39.6 k-in,0.67,OK
3,3b. sign post (std VPF post),51.3 k-in,39.6 k-in,1.29,NG
4,3c. sign post (Pipe4 upsize),51.3 k-in,95.4 k-in,0.54,OK
5,4. fabric tension-band bolts,68 lb,2237 lb,0.03,OK
6,"5a. anchors, bare fence",6620 lb,concrete governs,0.93,OK
7,"5b. anchors, fence + sign",12816 lb,concrete governs,1.80,NG
8,5c. wrap plate vs ODOT 7.1k,10.3 kips,7.1 kips,1.44,NG


## Conclusions

- **The standard VPF detail works for what it was designed for.**  Post,
  bands, and anchors all clear the bare-fence demand (the anchors at DCR ≈
  0.9 — no idle margin), and the wrap-plate demand sits at 75% of ODOT's
  published 7.1-kip design envelope.
- **The 3×5 solid sign breaks the fence in two places:** the 2.880-in post
  (flexure) and, more stubbornly, the **concrete side of the anchorage** —
  breakout/bond on a 3-in edge distance in a 12-in parapet top.  Upsizing the
  post is easy; upsizing the anchorage is not, because concrete governs and
  the VPF drawings expressly forbid swapping in mechanical anchors.
- Realistic paths for the sign: a **dedicated sign support** off the
  structure, a **special design** per VPF note (22)/(24) with anchor
  reinforcement tied into the parapet cage (new construction), or shrinking
  the sign/tightening post spacing until the standard envelope is respected.
- The remaining links to close out a real submission: the parapet itself and
  the deck overhang under the combined fence+sign line load (AASHTO LRFD BDS
  3.8.1.2.4 sound-barrier analogy, Section 13/A13.4), and the MASH
  crashworthiness question any barrier attachment raises.

**Fatigue note (LTS Section 11):** Fatigue I natural wind gust,
$P_{NW} = 5.2\,C_d\,I_F$ (Eq. 11.7.1.2-1):

In [13]:
P_nw = 5.2 * Cd_sign                      # Eq. 11.7.1.2-1, IF = 1.0
M_nw = P_nw * A_sign * z_sign_cg * 12     # sign only (dominant), lb-in
sr_std = M_nw / S_p / 1000                # on the standard 2.880 post
sr_big = M_nw / S_big / 1000              # on the Pipe4 upsize
print(f"P_NW = {P_nw:.1f} psf -> stress range: std post {sr_std:.1f} ksi, Pipe4 sign post {sr_big:.1f} ksi")
print("CAFT: ~4.5 ksi (Cat E', welded tube-to-base-plate), 7 ksi (Cat D, anchor rods)")
print("-> one more reason the standard post can't take the sign (fatigue NG even if")
print("   flexure were fixed); the Pipe4 sign post clears it comfortably")

P_NW = 6.1 psf -> stress range: std post 5.6 ksi, Pipe4 sign post 1.6 ksi
CAFT: ~4.5 ksi (Cat E', welded tube-to-base-plate), 7 ksi (Cat D, anchor rods)
-> one more reason the standard post can't take the sign (fatigue NG even if
   flexure were fixed); the Pipe4 sign post clears it comfortably


---
### References

- AASHTO *LRFD Specifications for Structural Supports for Highway Signs,
  Luminaires, and Traffic Signals* (LRFDLTS-1), 1st Ed. w/ interims — Arts.
  3.4, 3.8, 5.15, 5.16, 11.7
- ODOT SCD **VPF-1-24** (2024) and **VPF-1-90** (1990, rev. 2011) — Vandal
  Protection Fence
- ICC-ES **ESR-3187** — Hilti HIT-HY 200 adhesive anchors (Tables 10, 12, 14)
- ACI 318-19 Ch. 17 via `civilpy.structural.concrete.AnchorBolts`
- CLFMI *Wind Load Guide* — chain-link fabric net-area treatment